In [2]:
import os
import torch

BASE_DIR = "../model6/layers_output"

MODELS = {
    "orig": os.path.join(BASE_DIR, "orig_mamba"),
    "ssm": os.path.join(BASE_DIR, "ssm_mamba"),
    "gate": os.path.join(BASE_DIR, "gate_mamba"),
}

ATOL = 1e-6
RTOL = 1e-5


def compare_tensors(t1, t2):
    diff = (t1 - t2).abs()
    max_abs = diff.max().item()
    mean_abs = diff.mean().item()

    denom = t1.abs().clamp(min=1e-8)
    rel_diff = (diff / denom).max().item()

    allclose = torch.allclose(t1, t2, atol=ATOL, rtol=RTOL)

    return {
        "max_abs": max_abs,
        "mean_abs": mean_abs,
        "max_rel": rel_diff,
        "allclose": allclose
    }


def compare_model_pair(modelA_name, modelB_name):
    print(f"\n==============================")
    print(f"Comparing {modelA_name} vs {modelB_name}")
    print(f"==============================")

    modelA_dir = MODELS[modelA_name]
    modelB_dir = MODELS[modelB_name]

    subfolders = sorted(os.listdir(modelA_dir))

    for sub in subfolders:
        dirA = os.path.join(modelA_dir, sub)
        dirB = os.path.join(modelB_dir, sub)

        if not os.path.exists(dirB):
            continue

        print(f"\n--- Subfolder: {sub} ---")

        files = sorted(os.listdir(dirA))

        for fname in files:
            pathA = os.path.join(dirA, fname)
            pathB = os.path.join(dirB, fname)

            if not os.path.exists(pathB):
                continue

            tA = torch.load(pathA, map_location="cpu")
            tB = torch.load(pathB, map_location="cpu")

            stats = compare_tensors(tA, tB)

            if not stats["allclose"]:
                print(f"{fname}")
                print(f"  max_abs_diff:  {stats['max_abs']:.6e}")
                print(f"  mean_abs_diff: {stats['mean_abs']:.6e}")
                print(f"  max_rel_diff:  {stats['max_rel']:.6e}")
                print(f"  allclose:      {stats['allclose']}")

compare_model_pair("orig", "ssm")
compare_model_pair("orig", "gate")


Comparing orig vs ssm

--- Subfolder: contextualized_states ---
contextualized_states_layer_0.pt
  max_abs_diff:  2.792731e+00
  mean_abs_diff: 2.548022e-02
  max_rel_diff:  1.602296e+05
  allclose:      False
contextualized_states_layer_1.pt
  max_abs_diff:  1.718430e+00
  mean_abs_diff: 6.803352e-02
  max_rel_diff:  2.048503e+06
  allclose:      False
contextualized_states_layer_10.pt
  max_abs_diff:  4.850091e-01
  mean_abs_diff: 2.291552e-02
  max_rel_diff:  1.033127e+06
  allclose:      False
contextualized_states_layer_11.pt
  max_abs_diff:  4.827888e-01
  mean_abs_diff: 2.512603e-02
  max_rel_diff:  9.241307e+05
  allclose:      False
contextualized_states_layer_12.pt
  max_abs_diff:  5.564938e-01
  mean_abs_diff: 2.552150e-02
  max_rel_diff:  2.240042e+05
  allclose:      False
contextualized_states_layer_13.pt
  max_abs_diff:  5.238552e-01
  mean_abs_diff: 2.610863e-02
  max_rel_diff:  1.832737e+06
  allclose:      False
contextualized_states_layer_14.pt
  max_abs_diff:  5.26

EOFError: 

In [ ]:
!conda install decorator

import sys
print(sys.executable)
!conda activate falcon-gpu

: 

In [11]:
import os
import torch
import re
import plotly.graph_objects as go
from collections import defaultdict

BASE_DIR = "../model6/layers_output"

MODELS = {
    "orig": os.path.join(BASE_DIR, "orig_mamba"),
    "ssm": os.path.join(BASE_DIR, "ssm_mamba"),
    "gate": os.path.join(BASE_DIR, "gate_mamba"),
}

ATOL = 1e-6
RTOL = 1e-5


def extract_layer_idx(filename):
    match = re.search(r"layer_(\d+)", filename)
    return int(match.group(1)) if match else None


def compare_tensors(t1, t2):
    diff = (t1 - t2).abs()
    return diff.max().item()

# def compare_tensors(t1, t2):
#     diff = (t1 - t2).abs()
#     denom = t1.abs().clamp(min=1e-8)  # avoid division by zero
#     percent_diff = (diff / denom) * 100
#     return percent_diff.max().item()


# def compare_tensors(t1, t2):
#     t1 = t1.float()
#     t2 = t2.float()

#     numerator = torch.norm(t1 - t2)
#     denominator = torch.norm(t1).clamp(min=1e-8)

#     return (numerator / denominator).item() * 100

def compare_and_collect(modelA_name, modelB_name):
    print(f"\nCollecting stats: {modelA_name} vs {modelB_name}")

    modelA_dir = MODELS[modelA_name]
    modelB_dir = MODELS[modelB_name]

    results = defaultdict(dict)  # results[subfolder][layer_idx] = max_abs_diff

    subfolders = sorted(os.listdir(modelA_dir))

    for sub in subfolders:
        dirA = os.path.join(modelA_dir, sub)
        dirB = os.path.join(modelB_dir, sub)

        if not os.path.exists(dirB):
            continue

        files = sorted(os.listdir(dirA))

        for fname in files:
            layer_idx = extract_layer_idx(fname)
            if layer_idx is None:
                continue

            pathA = os.path.join(dirA, fname)
            pathB = os.path.join(dirB, fname)

            if not os.path.exists(pathB):
                continue

            tA = torch.load(pathA, map_location="cpu")
            tB = torch.load(pathB, map_location="cpu")

            max_diff = compare_tensors(tA, tB)
            results[sub][layer_idx] = max_diff

    return results


def plot_results(results, title):
    fig = go.Figure()

    for subfolder, layer_dict in results.items():
        layers = sorted(layer_dict.keys())
        diffs = [layer_dict[i] for i in layers]

        fig.add_trace(
            go.Scatter(
                x=layers,
                y=diffs,
                mode="lines+markers",
                name=subfolder
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Layer Index",
        yaxis_title="Max Absolute Difference",
        # yaxis_type="log",  # important to see small differences
        template="plotly_white",
        height=700,
    )

    fig.show()


# Run comparisons
results_ssm = compare_and_collect("orig", "ssm")
results_gate = compare_and_collect("orig", "gate")

# print("Gate vs SSM")
# print(results_gate)
# print("SSM vs Gate")
# print(results_ssm)

# Plot
plot_results(results_ssm, "Original vs SSM")
plot_results(results_gate, "Original vs Gate")